# Phase 1 — Explorer le référentiel, sans rien corriger 

In [6]:
import pandas as pd
donnees = pd.read_csv("../data/numeros_tva.csv",keep_default_na=False)
#, na_values= "-"

### 1.1 — Combien de pays distincts ? Combien de formats différents pour un même pays ?

In [7]:
donnees["pays_declare"].nunique()
display(donnees["pays_declare"].unique())
print(donnees["pays_declare"].value_counts())


<StringArray>
['DK', 'PT', 'LU', 'SE', 'NL', 'BE', 'FI', 'GB', 'FR', 'ZZ', 'PL', 'IT', 'UK',
 'XX', 'QQ']
Length: 15, dtype: str

pays_declare
FR    974
DK    961
BE    959
LU    956
SE    949
PT    947
NL    947
IT    947
PL    939
FI    902
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64


Il y a 15 codes de pays distincts, dont 5 codes "suspects" représentant 519 lignes. Ces anomalies pourraient provenir d'erreurs de saisie et nécessitent une vérification.

### 1.2 — Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?

In [8]:
donnees["numero_tva"].head(20)

0            DK73224645
1           PT787327239
2            LU71043234
3       SE.751298146001
4        NL505862176B88
5            DK61530843
6       NL505862176B88 
7       SE.314125231601
8            DK48646344
9          BE1997422119
10          FI.84564443
11       NL767957817B57
12      SE-353056663101
13      SE1861093621016
14          PT901258090
15       NL962988030B37
16         GB0749640348
17         BE1628475293
18         BE0605238824
19        FR63432337650
Name: numero_tva, dtype: str

Les numéros de TVA présentent plusieurs formats différents : certains contiennent des points, des tirets et d'autres n'ont rien. Ces différences peuvent provoquer des erreurs lors de la vérification avec l'API. 

### 1.3 — Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?

In [ ]:
#Verifier que keep_default_na=False fonctionne
print("ID =",donnees["id"].isna().sum(),
      "Raison social = ", donnees["raison_sociale"].isna().sum(),
      "Pays déclaré = ", donnees["pays_declare"].isna().sum(),
      "Numero tva = ", donnees["numero_tva"].isna().sum(),
      "Date saisie = ", donnees["date_saisie"].isna().sum(),
      "Source saisie = ", donnees["source_saisie"].isna().sum(),
)
display("null =",(donnees == "null").sum())
display("NULL =",(donnees == "NULL").sum())
display("- =",(donnees == "-").sum())
display("'' =",(donnees == "").sum())
display(". =",(donnees == ".").sum())
display("' '=",(donnees == " ").sum())
display("NaN =",(donnees == "NaN").sum())
display("nan =",(donnees == "nan").sum())
display("N/A =",(donnees == "N/A").sum())





ID = 0 Raison social =  0 Pays déclaré =  0 Numero tva =  0 Date saisie =  0 Source saisie =  0


'null ='

id                 0
raison_sociale     0
pays_declare       0
numero_tva        38
date_saisie        0
source_saisie      0
dtype: int64

'NULL ='

id                0
raison_sociale    0
pays_declare      0
numero_tva        0
date_saisie       0
source_saisie     0
dtype: int64

'- ='

... (9 outputs left out)

'nan ='

id                0
raison_sociale    0
pays_declare      0
numero_tva        0
date_saisie       0
source_saisie     0
dtype: int64

'N/A ='

id                 0
raison_sociale     0
pays_declare       0
numero_tva        48
date_saisie        0
source_saisie      0
dtype: int64

J'ai identifié 260 valeurs vides dans la colonne numero_tva, sous 5 formes différentes : chaîne vide, espace, tiret, null et N/A.

### 1.4 — Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?

In [19]:
prefixe_tva = donnees["numero_tva"].str[0:2]
verif_cmr = (prefixe_tva != donnees["pays_declare"])
display(donnees[verif_cmr][["numero_tva", "pays_declare"]])
donnees[verif_cmr][["pays_declare"]].value_counts()

,numero_tva,pays_declare
6,NL505862176B88,NL
22,FR58658124710,FR
25,-,PT
30,PT682102423,PT
37,FI26606369,FI
...,...,...
9955,63761613,DK
9960,866309250,PT
9963,dk62994096,DK
9984,fi30649863,FI


pays_declare
PL              176
FR              175
NL              169
LU              168
SE              165
BE              164
DK              159
PT              158
IT              152
FI              149
ZZ                4
UK                3
QQ                2
XX                1
Name: count, dtype: int64

Non, le pays déclaré dans le CRM ne correspond pas toujours au préfixe du numéro de TVA. Sur 10 000 enregistrements, 1 645 présentent une différence. Les pays les plus représentés parmi ces différences sont la Pologne (176), la France (175) et les Pays-Bas (169).

### 1.5 — Le CSV et l'Excel contiennent-ils vraiment la même chose une fois chargés ? Lequel prenez-vous comme source de vérité, et pourquoi ?

In [10]:
prefixe_tva = donnees["numero_tva"].str[0:2]
print((prefixe_tva != donnees["pays_declare"]).sum())
data_csv = pd.read_csv("../data/numeros_tva.csv", dtype=str, keep_default_na=False)
data_xlsx = pd.read_excel("../data/numeros_tva.xlsx", dtype=str, keep_default_na=False)
for col in data_csv.columns:
    differences = data_csv[col] != data_xlsx[col]
    nb_diff = differences.sum()
    print(f"Colonne '{col}' : {nb_diff} différences")
    if nb_diff > 0:
        comparaison = pd.DataFrame({
            "id": data_csv.loc[differences, "id"],
            "CSV": data_csv.loc[differences, col],
            "Excel": data_xlsx.loc[differences, col]
        })
        print(comparaison.head(5))
        print("-" * 40) 

1645
Colonne 'id' : 0 différences
Colonne 'raison_sociale' : 0 différences
Colonne 'pays_declare' : 0 différences
Colonne 'numero_tva' : 0 différences
Colonne 'date_saisie' : 0 différences
Colonne 'source_saisie' : 0 différences


Après comparaison des deux fichiers avec pandas, aucune différence n'a été détectée entre les colonnes.
Je choisis le fichier CSV comme source de vérité pour mon analyse, car il est simple à lire avec pandas, ne contient pas de mise en forme et ne nécessite pas de bibliothèque supplémentaire

### 1.6 — Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?

In [18]:
display(donnees["source_saisie"].value_counts())
date = donnees["date_saisie"].str[0:7].value_counts()
print(date.sort_values(ascending=False))

source_saisie
reprise_erp           2071
crm                   2019
portail_client        1990
saisie_manuelle       1976
import_fournisseur    1944
Name: count, dtype: int64

date_saisie
2024-04    510
2025-08    508
2024-03    504
2024-06    503
2025-07    500
2025-01    497
2024-12    494
2024-11    494
2024-05    489
2024-08    489
2024-07    469
2025-06    468
2025-04    465
2024-01    461
2024-10    458
2025-05    457
2025-03    456
2024-02    446
2025-09    445
2024-09    443
2025-02    428
2025-10     16
Name: count, dtype: int64


Les cinq canaux de saisie sont assez équilibrés et les saisies sont relativement régulières d'un mois à l'autre, à l'exception du mois d'octobre 2025, où l'on ne compte que 16 saisies.

**CONCLUSION :** 
- 260 valeurs manquantes dans les numéros de TVA.
- 1 645 différences entre les préfixes TVA et les pays déclarés.
- 0 différence entre les fichiers CSV et Excel.
- 5 canaux de saisie assez équilibrés.
- Des saisies relativement régulières, sauf en octobre 2025, avec seulement 16 enregistrements.
- 15 codes pays distincts dans le CRM, alors que 10 étaient attendus. Certains codes, comme ZZ, QQ et XX, semblent incorrects. Les codes GB et UK sont également présents et devront être vérifiés. Il peut donc y avoir des erreurs de saisie.